In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import ast
import random
import math
from collections import defaultdict
from tqdm import tqdm

In [3]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/sampled_df.csv")

print(df.shape)
df.head()

(2107420, 16)


,userId,movie_id,rating,tmdbId,genres,cast,keywords,director,overview,original_language,runtime,popularity,vote_average,vote_count,release_year,title
0,10007,11971,3.0,11971,"['Drama', 'Comedy', 'Romance']","['Kenneth Branagh', 'Richard Briers', 'Michael...","['shakespeare', 'bachelor', 'new love', 'rage ...",['Kenneth Branagh'],"In this Shakespearean farce, Hero and her groo...",en,111.0,8.627240,7.2,194.0,1993.0,Much Ado About Nothing
1,10007,1572,4.0,1572,"['Action', 'Thriller']","['Bruce Willis', 'Jeremy Irons', 'Samuel L. Ja...","['bomb', 'taxi', 'riddle', 'robbery', 'detecti...",['John McTiernan'],New York detective John McClane is back and ki...,en,128.0,15.138855,6.9,2094.0,1995.0,Die Hard: With a Vengeance
2,10007,197,5.0,197,"['Action', 'Drama', 'History', 'War']","['Mel Gibson', 'Catherine McCormack', 'Sophie ...","['individual', 'scotland', 'in love with enemy...",['Mel Gibson'],"Enraged at the slaughter of Murron, his new br...",en,177.0,20.755149,7.7,3404.0,1995.0,Braveheart
3,10007,268,3.0,268,"['Fantasy', 'Action']","['Jack Nicholson', 'Michael Keaton', 'Kim Basi...","['double life', 'dc comics', 'dual identity', ...",['Tim Burton'],The Dark Knight of Gotham City begins his war ...,en,126.0,19.106730,7.0,2145.0,1989.0,Batman
4,10007,274,4.0,274,"['Crime', 'Drama', 'Thriller']","['Jodie Foster', 'Anthony Hopkins', 'Scott Gle...","['based on novel', 'psychopath', 'horror', 'su...",['Jonathan Demme'],"FBI trainee, Clarice Starling ventures into a ...",en,119.0,4.307222,8.1,4549.0,1991.0,The Silence of the Lambs


In [4]:
def safe_literal_eval(val):
    """Return list from string representation, else [] on failure."""
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except Exception:
            return []
    return val if isinstance(val, list) else []

def ensure_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [5]:
print("Converting stringified lists back to list objects...")
list_columns = ['genres', 'cast', 'keywords', 'director']
for col in list_columns:
    df[col] = df[col].apply(safe_literal_eval)

Converting stringified lists back to list objects...


In [6]:
#in order to make one big string for the overview,cast,title,keyword,director and we embbed this string
def build_movie_text(row):
    parts = []

    if pd.notna(row['title']):
        parts.append(f"Title: {row['title']}")

    if pd.notna(row['overview']):
        parts.append(f"Overview: {row['overview']}")

    cast = safe_literal_eval(row['cast'])
    if isinstance(cast, list) and len(cast) > 0:
        parts.append("Cast: " + ", ".join(cast[:5]))

    director = safe_literal_eval(row['director'])
    if isinstance(director, list) and len(director) > 0:
        parts.append("Director: " + ", ".join(director))

    keywords = safe_literal_eval(row['keywords'])
    if isinstance(keywords, list) and len(keywords) > 0:
        parts.append("Keywords: " + ", ".join(keywords[:10]))

    return ". ".join(parts)


df["full_text"] = df.apply(build_movie_text, axis=1)



In [7]:
print(df.loc[0, "full_text"])

Title: Much Ado About Nothing. Overview: In this Shakespearean farce, Hero and her groom-to-be, Claudio, team up with Claudio's commanding officer, Don Pedro, the week before their wedding to hatch a matchmaking scheme. Their targets are sharp-witted duo Benedick and Beatrice -- a tough task indeed, considering their corresponding distaste for love and each other. Meanwhile, meddling Don John plots to ruin the wedding.. Cast: Kenneth Branagh, Richard Briers, Michael Keaton. Director: Kenneth Branagh. Keywords: shakespeare, bachelor, new love, rage and hate, lover, wedding, intrigue


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2107420 entries, 0 to 2107419
Data columns (total 17 columns):
 #   Column             Dtype  
---  ------             -----  
 0   userId             int64  
 1   movie_id           int64  
 2   rating             float64
 3   tmdbId             int64  
 4   genres             object 
 5   cast               object 
 6   keywords           object 
 7   director           object 
 8   overview           object 
 9   original_language  object 
 10  runtime            float64
 11  popularity         float64
 12  vote_average       float64
 13  vote_count         float64
 14  release_year       float64
 15  title              object 
 16  full_text          object 
dtypes: float64(6), int64(3), object(8)
memory usage: 273.3+ MB


In [9]:
!pip install -q pandas numpy torch scikit-learn sentence-transformers tf-keras


In [10]:


import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer

# Config / Hyperparams

SEED = 42
BATCH_SIZE = 512
EMBED_DIM = 64
LR = 1e-3
EPOCHS = 25
NEGATIVE_SAMPLES = 100
TEXT_EMB_MODEL = "all-MiniLM-L6-v2"
EMBED_CACHE_DIR = "./emb_cache"
TOPK = [10, 20]            # metrics to compute (Recall@K, NDCG@K)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [11]:
# 1) Prepare / Clean Data
# This part makes the data ready for training.
# - Make userId and movie_id strings
# - Turn list-like columns (genres, cast, keywords, director) into Python lists
# - Keep only one row per movie (unique movies)
# - Build a map from movie_id to index
# - Count how many unique movies and users exist
# - Build maps for userId from to index

print("Preparing data...")


df['userId'] = df['userId'].astype(str)
df['movie_id'] = df['movie_id'].astype(str)

list_fields = ['genres', 'cast', 'keywords', 'director']
for f in list_fields:
    if f in df.columns:
        df[f] = df[f].apply(safe_literal_eval)
    else:
        df[f] = [[] for _ in range(len(df))]

movies_df = df.drop_duplicates(subset=['movie_id']).reset_index(drop=True).copy()
movies_df_index_map = {mid: idx for idx, mid in enumerate(movies_df['movie_id'].astype(str).tolist())}
num_items = len(movies_df)
print(f"Unique movies: {num_items}")

unique_users = df['userId'].unique().tolist()
user2idx = {u: i for i, u in enumerate(unique_users)}
idx2user = {i: u for u, i in user2idx.items()}
num_users = len(unique_users)
print(f"Unique users: {num_users}")



Preparing data...
Unique movies: 12988
Unique users: 17000


In [12]:
# 2) Feature Engineering for Items
# This part builds feature vectors for each movie (item).
# - Genres: turn list of genres into multi-hot vectors (dim = 20 we have 20 genre )
# - Numericals: scale runtime, popularity, vote_average, vote_count, release_year (dim = 5) we used minmax scaling
# - Text: create embeddings from fullt_ext using SentenceTransformer (dim = 384 output of "all-MiniLM-L6-v2" model )
# - Cache text embeddings to avoid recompute
# - Combine all parts into one dense vector per item
#   Final shape: (num_items, D_item)
#   where D_item = dim_genres + dim_numerical + dim_text torch.Size([11532, 409])
# - Save as torch tensor for training

print("\nProcessing item features...")

# 2.1 Genres (multi hot)
mlb = MultiLabelBinarizer(sparse_output=False)
genres_lists = movies_df['genres'].apply(lambda x: x if isinstance(x, list) else []).tolist()
genres_encoded = mlb.fit_transform(genres_lists) if len(genres_lists) else np.zeros((len(movies_df), 0))
genres_df = pd.DataFrame(genres_encoded, columns=mlb.classes_, index=movies_df.index)

# 2.2 Numerical features scaling
numerical_cols = [c for c in ['runtime', 'popularity', 'vote_average', 'vote_count', 'release_year'] if c in movies_df.columns]
scaler = MinMaxScaler()
if numerical_cols:
    movies_df[numerical_cols] = movies_df[numerical_cols].fillna(0)
    num_feats = scaler.fit_transform(movies_df[numerical_cols])
else:
    num_feats = np.zeros((len(movies_df), 0))

# 2.3 Text embeddings
ensure_dir(EMBED_CACHE_DIR)
fulltext_cache_path = os.path.join(EMBED_CACHE_DIR, "fulltext_emb.npy")

text_model = SentenceTransformer(TEXT_EMB_MODEL)

if os.path.exists(fulltext_cache_path):
    print("Loading cached full_text embeddings...")
    fulltext_embeddings = np.load(fulltext_cache_path)
    assert fulltext_embeddings.shape[0] == len(movies_df)
else:
    print("Computing full_text embeddings (this may take a while)...")
    full_texts = movies_df['full_text'].fillna('').astype(str).tolist()
    fulltext_embeddings = text_model.encode(full_texts, show_progress_bar=True, convert_to_numpy=True)
    np.save(fulltext_cache_path, fulltext_embeddings)
    print("Saved full_text embeddings to cache.")

# 2.4 Combine item features into a numpy array for all items
print("Combining item features into dense vectors...")
item_feature_list = []
for idx in range(len(movies_df)):
    parts = []
    if genres_df.shape[1] > 0:
        parts.append(genres_df.iloc[idx].values.astype(np.float32))
    if num_feats.shape[1] > 0:
        parts.append(num_feats[idx].astype(np.float32))
    parts.append(fulltext_embeddings[idx].astype(np.float32))
    item_vec = np.concatenate(parts).astype(np.float32)
    item_feature_list.append(item_vec)
item_features = np.stack(item_feature_list)
item_input_dim = item_features.shape[1]
print(f"Item feature dim: {item_input_dim}")

item_features_tensor = torch.from_numpy(item_features)



Processing item features...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading cached full_text embeddings...
Combining item features into dense vectors...
Item feature dim: 409


In [13]:
item_features_tensor.shape

torch.Size([12988, 409])

In [14]:
# 3) Build user positive interactions (for negative sampling & training)
# - Mark ratings >= 4.0 as positive so user have rate movie more than 4 its a good movie for the user
# - Keep only valid users and movies
# - Map userId to user_idx, movie_id to item_idx
# - Build dictionary: user_idx to set of item_idx (positives)
# - Drop users with no positive interactions
# - Make a list of all item indices (for sampling negatives later)
#
# Example:
#   user_pos_items = {
#       0: {2, 15, 37},    # user 0 liked items 2, 15, 37
#       1: {4, 9},         # user 1 liked items 4, 9
#       2: {0, 5, 8, 22},  # user 2 liked items 0, 5, 8, 22
#       ...
#   }
#   all_item_indices = [0, 1, 2, ..., num_items-1]

print("\nBuilding user -> positive items mapping...")


POS_THRESHOLD = 4.0
pos_df = df[df['rating'] >= POS_THRESHOLD].copy()
pos_df = pos_df[pos_df['movie_id'].isin(movies_df['movie_id'].astype(str))]
pos_df['user_idx'] = pos_df['userId'].map(user2idx)
pos_df['item_idx'] = pos_df['movie_id'].map(movies_df_index_map)

user_pos_items = defaultdict(set)
for _, r in pos_df.iterrows():
    user_pos_items[int(r['user_idx'])].add(int(r['item_idx']))

valid_users = [u for u in range(num_users) if len(user_pos_items[u]) > 0]
print(f"Users with >=1 positive: {len(valid_users)}")

all_item_indices = np.arange(num_items, dtype=np.int64)




Building user -> positive items mapping...
Users with >=1 positive: 16888


In [15]:
# 4) Dataset for training with negative sampling
#    returns: user_idx, pos_item_idx, (neg_item_indices)
class RetrievalTrainDataset(Dataset):
    def __init__(self, user_pos_items, num_items, neg_samples=100, users_list=None):
        """
        user_pos_items: dict user_idx -> set(item_idx)
        num_items: total number of items
        neg_samples: number of negatives to sample per positive
        users_list: optional list of users to include (for splitting)
        """
        self.user_pos_items = user_pos_items
        self.num_items = num_items
        self.neg_samples = neg_samples
        self.entries = []
        ulist = users_list if users_list is not None else list(user_pos_items.keys())
        for u in ulist:
            for pos in user_pos_items[u]:
                self.entries.append((u, pos))

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        u, pos_i = self.entries[idx]
        negs = []
        while len(negs) < self.neg_samples:
            cand = random.randint(0, self.num_items - 1)
            if cand not in self.user_pos_items[u]:
                negs.append(cand)
        return np.int64(u), np.int64(pos_i), np.array(negs, dtype=np.int64)


In [16]:
# 5) Split users into train / val

print("\nSplitting users into train/val (user-wise split)...")
train_users, val_users = train_test_split(valid_users, test_size=0.2, random_state=SEED)
print(f"Train users: {len(train_users)}, Val users: {len(val_users)}")

train_dataset = RetrievalTrainDataset(user_pos_items, num_items, neg_samples=NEGATIVE_SAMPLES, users_list=train_users)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

val_user_pos_items = {u: user_pos_items[u] for u in val_users}


Splitting users into train/val (user-wise split)...
Train users: 13510, Val users: 3378


In [17]:
# 6) Model: Two-tower (MLP heads). We'll produce L2-normalized embeddings and compute dot-products.

print("\nBuilding model...")

class MLP(nn.Module):
    def __init__(self, input_dim, hidden=[256, 128], output_dim=64):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h in hidden:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.ReLU())
            in_dim = h
        layers.append(nn.Linear(in_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class TwoTowerRetrieval(nn.Module):
    def __init__(self, user_input_dim, item_input_dim, emb_dim=64):
        super().__init__()
        self.user_tower = MLP(user_input_dim, hidden=[256,128], output_dim=emb_dim)
        self.item_tower = MLP(item_input_dim, hidden=[256,128], output_dim=emb_dim)

    def forward_user(self, user_features):
        return self.user_tower(user_features)

    def forward_item(self, item_features):
        return self.item_tower(item_features)

    def l2_normalize(self, x, eps=1e-8):
        return x / (x.norm(p=2, dim=1, keepdim=True) + eps)

model = TwoTowerRetrieval(item_input_dim, item_input_dim, emb_dim=EMBED_DIM).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

criterion = nn.CrossEntropyLoss()


# Callbacks: LR scheduler, EarlyStopping, Checkpoint

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2
)

class EarlyStopping:
    def __init__(self, patience=5):
        self.patience = patience
        self.best_score = None
        self.counter = 0
        self.early_stop = False

    def step(self, score):
        if self.best_score is None or score > self.best_score:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

early_stopping = EarlyStopping(patience=5)
best_recall10 = 0.0



Building model...


In [18]:
# 7) Helper: Create a user feature from positive item features (same approach you had: mean of positives)
#    We'll precompute and cache user features for both train and val users (on CPU).
print("\nPrecomputing user feature vectors (mean of their positive item features)...")

item_feats_np = item_features
user_features_np = np.zeros((num_users, item_feats_np.shape[1]), dtype=np.float32)

for u in tqdm(range(num_users)):
    pos_items = list(user_pos_items.get(u, []))
    if len(pos_items) > 0:
        user_features_np[u] = item_feats_np[pos_items].mean(axis=0)
    else:
        user_features_np[u] = item_feats_np.mean(axis=0)

user_features_tensor = torch.from_numpy(user_features_np).to(DEVICE)


Precomputing user feature vectors (mean of their positive item features)...


100%|██████████| 17000/17000 [00:01<00:00, 11982.99it/s]


In [19]:
# 8) Training loop with negative sampling (InfoNCE-like)
#    For each batch:
#      - get batch of users (B), pos_items (B), negs (B, K)
#      - compute user embeddings (B, E)
#      - compute item embeddings for pos and negs: (B, 1+K, E)
#      - compute logits = user_emb @ item_emb.T -> (B, 1+K)
#      - apply temperature scaling if desired
#      - cross-entropy with target zeros
TEMPERATURE = 0.05

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    it = 0
    for batch in tqdm(loader, desc="Training batches"):
        users_np, pos_np, negs_np = batch
        users = torch.tensor(users_np, dtype=torch.long, device=device)
        pos_items = torch.tensor(pos_np, dtype=torch.long, device=device)
        neg_items = torch.tensor(negs_np, dtype=torch.long, device=device)

        B = users.shape[0]
        K = neg_items.shape[1]

        user_feats = user_features_tensor[users]
        item_pos_feats = item_features_tensor[pos_items].to(device)
        item_neg_feats = item_features_tensor[neg_items].to(device)

        user_emb = model.forward_user(user_feats)
        pos_emb = model.forward_item(item_pos_feats)
        neg_emb = model.forward_item(item_neg_feats.view(-1, item_neg_feats.shape[-1])).view(B, K, -1)

        user_emb = model.l2_normalize(user_emb)
        pos_emb = model.l2_normalize(pos_emb)
        neg_emb = model.l2_normalize(neg_emb.view(B*K, -1)).view(B, K, -1)

        pos_logits = torch.sum(user_emb * pos_emb, dim=1, keepdim=True)
        neg_logits = torch.bmm(neg_emb, user_emb.unsqueeze(2)).squeeze(2)
        logits = torch.cat([pos_logits, neg_logits], dim=1) / TEMPERATURE

        labels = torch.zeros(B, dtype=torch.long, device=device)

        loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        it += 1
    avg_loss = total_loss / max(1, it)
    return avg_loss


In [20]:
# 9) Validation metrics
#    For each user in val_user_pos_items:
#      - compute user embedding
#      - compute dot product with all item embeddings (or with subset to reduce compute)
#      - remove items in user's train positives if you want strict eval (we didn't split interactions by time, but we split users)
#      - get top-k and compute Recall@K and NDCG@K

@torch.no_grad()
def compute_user_and_item_embeddings(model, item_features_tensor, device, batch_size=2048):
    model.eval()
    item_feats = item_features_tensor.to(device)
    all_item_embs = []
    with torch.no_grad():
        for i in range(0, item_feats.shape[0], batch_size):
            chunk = item_feats[i:i+batch_size].to(device)
            emb = model.forward_item(chunk)
            emb = model.l2_normalize(emb)
            all_item_embs.append(emb.cpu())
    all_item_embs = torch.cat(all_item_embs, dim=0)
    user_feats_all = user_features_tensor.to(device)
    user_embs_all = model.forward_user(user_feats_all)
    user_embs_all = model.l2_normalize(user_embs_all).cpu()
    return user_embs_all, all_item_embs

def recall_at_k(predicted, ground_truth, k):
    pred_k = predicted[:k]
    hits = len(set(pred_k).intersection(set(ground_truth)))
    return hits / len(ground_truth)

def dcg_at_k(predicted, ground_truth, k):
    pred_k = predicted[:k]
    dcg = 0.0
    for i, p in enumerate(pred_k):
        if p in ground_truth:
            dcg += 1.0 / math.log2(i + 2)
    return dcg

def idcg_at_k(ground_truth, k):
    gt_len = min(len(ground_truth), k)
    idcg = sum((1.0 / math.log2(i + 2) for i in range(gt_len)))
    return idcg

def ndcg_at_k(predicted, ground_truth, k):
    dcg = dcg_at_k(predicted, ground_truth, k)
    idcg = idcg_at_k(ground_truth, k)
    return dcg / idcg if idcg > 0 else 0.0

@torch.no_grad()
def evaluate_recall_ndcg(model, val_user_pos_items, user2idx_map, item_embs, user_embs, topk_list=[10,20], device=torch.device('cpu')):
    """
    user_embs: tensor(num_users, E) on CPU
    item_embs: tensor(num_items, E) on CPU
    val_user_pos_items: dict user_idx -> set(pos_item_idxs)
    """
    model.eval()
    item_embs = item_embs.to(device)
    user_embs = user_embs.to(device)

    recall_results = {k: [] for k in topk_list}
    ndcg_results = {k: [] for k in topk_list}

    val_users = list(val_user_pos_items.keys())
    BATCH_U = 256
    for start in range(0, len(val_users), BATCH_U):
        batch_users = val_users[start:start+BATCH_U]
        u_idxs = torch.tensor(batch_users, dtype=torch.long, device=device)
        u_emb = user_embs[u_idxs]
        scores = torch.matmul(u_emb, item_embs.t())
        for i, u in enumerate(batch_users):
            scores_u = scores[i]
            topk_vals, topk_idx = torch.topk(scores_u, k=max(topk_list))
            topk_idx = topk_idx.cpu().numpy().tolist()
            gt = list(val_user_pos_items[u])
            for k in topk_list:
                r = recall_at_k(topk_idx, gt, k)
                n = ndcg_at_k(topk_idx, gt, k)
                recall_results[k].append(r)
                ndcg_results[k].append(n)

    recall_avg = {k: np.mean(recall_results[k]) if len(recall_results[k]) else 0.0 for k in topk_list}
    ndcg_avg = {k: np.mean(ndcg_results[k]) if len(ndcg_results[k]) else 0.0 for k in topk_list}
    return recall_avg, ndcg_avg



In [21]:
# 10) Training + Validation loop

print("\nStarting training...")
item_features_tensor = item_features_tensor.to(DEVICE)

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    print(f"Epoch {epoch+1}/{EPOCHS} => Train Loss: {train_loss:.4f}")

    print("Computing embeddings for evaluation...")
    user_embs_all, item_embs_all = compute_user_and_item_embeddings(model, item_features_tensor.cpu(), DEVICE, batch_size=2048)
    print("Evaluating Recall@K and NDCG@K on validation users...")
    recall_avg, ndcg_avg = evaluate_recall_ndcg(model, val_user_pos_items, user2idx, item_embs_all.cpu(), user_embs_all.cpu(), topk_list=TOPK, device=torch.device('cpu'))
    for k in TOPK:
        print(f"Val Recall@{k}: {recall_avg[k]:.4f}   Val NDCG@{k}: {ndcg_avg[k]:.4f}")


    # Step scheduler based on validation Recall@10
    scheduler.step(recall_avg[10])
    print(f"Current LR: {optimizer.param_groups[0]['lr']}")

    # Early stopping
    early_stopping.step(recall_avg[10])
    if early_stopping.early_stop:
        print("Early stopping triggered!")
        break

    # Checkpoint: save best model by Recall@10
    if recall_avg[10] > best_recall10:
        best_recall10 = recall_avg[10]
        torch.save(model.state_dict(), "best_model.pt")
        print("Saved new best model.")

print("Training complete.")



Starting training...


Training batches:   0%|          | 0/1577 [00:00<?, ?it/s]/tmp/ipython-input-981528480.py:19: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  users = torch.tensor(users_np, dtype=torch.long, device=device)
/tmp/ipython-input-981528480.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pos_items = torch.tensor(pos_np, dtype=torch.long, device=device)
/tmp/ipython-input-981528480.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  neg_items = torch.tensor(negs_np, dtype=torch.long, device=device)
Training batches: 100%|██████████| 1577/157

Epoch 1/25 => Train Loss: 2.3243
Computing embeddings for evaluation...
Evaluating Recall@K and NDCG@K on validation users...
Val Recall@10: 0.1744   Val NDCG@10: 0.4722
Val Recall@20: 0.2616   Val NDCG@20: 0.4542
Current LR: 0.001
Saved new best model.


Training batches: 100%|██████████| 1577/1577 [01:20<00:00, 19.62it/s]


Epoch 2/25 => Train Loss: 2.0566
Computing embeddings for evaluation...
Evaluating Recall@K and NDCG@K on validation users...
Val Recall@10: 0.1951   Val NDCG@10: 0.5062
Val Recall@20: 0.2881   Val NDCG@20: 0.4887
Current LR: 0.001
Saved new best model.


...

Epoch 23/25 => Train Loss: 1.6944
Computing embeddings for evaluation...
Evaluating Recall@K and NDCG@K on validation users...
Val Recall@10: 0.2752   Val NDCG@10: 0.6302
Val Recall@20: 0.3850   Val NDCG@20: 0.6111
Current LR: 0.001
Saved new best model.


Training batches: 100%|██████████| 1577/1577 [01:20<00:00, 19.53it/s]


Epoch 24/25 => Train Loss: 1.6883
Computing embeddings for evaluation...
Evaluating Recall@K and NDCG@K on validation users...
Val Recall@10: 0.2789   Val NDCG@10: 0.6344
Val Recall@20: 0.3892   Val NDCG@20: 0.6155
Current LR: 0.001
Saved new best model.


Training batches: 100%|██████████| 1577/1577 [01:20<00:00, 19.52it/s]


Epoch 25/25 => Train Loss: 1.6828
Computing embeddings for evaluation...
Evaluating Recall@K and NDCG@K on validation users...
Val Recall@10: 0.2778   Val NDCG@10: 0.6334
Val Recall@20: 0.3892   Val NDCG@20: 0.6150
Current LR: 0.001
Training complete.


In [23]:
import json
import numpy as np
import pandas as pd
from google.colab import files

print("Preparing files for download...")

print("Downloading best_model.pt...")
files.download('best_model.pt')

np.save('item_features.npy', item_features)
print("Downloading item_features.npy...")
files.download('item_features.npy')

movies_df_to_save = movies_df[['movie_id', 'title']]
movies_df_to_save.to_csv('movies_df.csv', index=False)
print("Downloading movies_df.csv...")
files.download('movies_df.csv')

np.save('user_features.npy', user_features_np)
print("Downloading user_features.npy...")
files.download('user_features.npy')

with open('user2idx.json', 'w') as f:
    json.dump(user2idx, f)
print("Downloading user2idx.json...")
files.download('user2idx.json')

print("\\nAll files have been prepared for download.")
print("Please check your browser's download manager.")

Preparing files for download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

\nAll files have been prepared for download.
Please check your browser's download manager.
